In [1]:
import kagglehub
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

# Download dataset
path = kagglehub.dataset_download(
    "eplfixturestoday/premier-league-team-all-time-table-19922026"
)

csv_path = os.path.join(
    path, "datasets", "team_season_records.csv"
)

df = pd.read_csv(csv_path)

#df = df[df["season"] == "2025-26"].copy()


# x = played; y=won
x = df['goals_against']
y = df['won']

#print(df.head())


model = LinearRegression()
x = df[['goal_difference']]
y = df['won']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(x_train, y_train)

print(f"Coefficient: {model.coef_[0].round(2)}\nIntercept: {model.intercept_.round(2)}")


prediction  = model.predict(x_test)

results = x_test.copy()

results['Actual wins'] = y_test
results['Predicted wins'] = prediction
results['Residual'] = results['Actual wins'] - results['Predicted wins']
print(results.head(5).round(2)) 

print(f'Mean squared error:{round(mean_squared_error(y_test,prediction),2)}')




#============================================#
#                OUTPUT                      #

#Coefficient: 0.22
#Intercept: 14.24
#the coefficient and intercept mean the regression formula will be wins = 0.22x+14.24
#this means that for every one increase in goal difference,the model will predict an average increase of 0.22 wins,
#  

#     goal_difference  Actual wins  Predicted wins  Residual
#286               56           28           26.46      1.54
#652               12           19           16.86      2.14
#257               -6           11           12.93     -1.93
#336              -11           12           11.84      0.16
#318              -10           10           12.06     -2.06
#Mean squared error:3.53


mae = mean_absolute_error(y_test, prediction)
mse = mean_squared_error(y_test, prediction)
rmse = np.sqrt(mse)
print(f"MAE : {mae:.2f} wins")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f} wins")
print(f"R2  : {r2_score(y_test, prediction):.3f}")
print(f"Baseline RMSE (always predict mean wins): {np.sqrt(mean_squared_error(y_test, [y_train.mean()] * len(y_test))):.2f}")

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_cv = -cross_val_score(model, x, y, cv=kf, scoring="neg_root_mean_squared_error")
print("CV RMSE folds:", rmse_cv.round(2), f"mean={rmse_cv.mean():.2f} std={rmse_cv.std():.2f}")


c:\Users\kgaogelo Tladi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Coefficient: 0.22
Intercept: 14.24
     goal_difference  Actual wins  Predicted wins  Residual
286               56           28           26.46      1.54
652               12           19           16.86      2.14
257               -6           11           12.93     -1.93
336              -11           12           11.84      0.16
318              -10           10           12.06     -2.06
Mean squared error:3.53
MAE : 1.37 wins
MSE : 3.53
RMSE: 1.88 wins
R2  : 0.879
Baseline RMSE (always predict mean wins): 5.40
CV RMSE folds: [1.88 2.03 1.89 1.8  2.08] mean=1.94 std=0.10
